In [1]:
"""
Interactive 2D Vector Playground  —  Python / Tkinter
======================================================
Operations:
  Add · Subtract · Scalar mult. · Unit vector
  Dot product · Orthogonal · Projection

Controls:
  • Drag the circle at any arrowhead to move that vector live
  • Type x / y values in the side panel
  • Choose an operation from the 8 buttons
  • k-slider & k-entry for scalar multiplication

Run:
    python vector_playground.py

Requirements: Python 3.8+  (stdlib only — tkinter is included)
Linux users may need:  sudo apt install python3-tk
"""

import tkinter as tk
import math

# ── Palette ───────────────────────────────────────────────────────────────────
BG        = "#1C1C2E"
GRID_MIN  = "#23233A"
GRID_MAJ  = "#2E2E4E"
AXIS_COL  = "#4A4A7A"
FG        = "#E8E6F0"
FG_DIM    = "#7A7A9A"
PANEL_BG  = "#12122A"
INPUT_BG  = "#22223A"
BTN_BG    = "#22223A"
BTN_ON_BG = "#0C447C"
BTN_ON_FG = "#85B7EB"

COLORS = ["#4FC3F7", "#EF476F", "#06D6A0", "#FFD166", "#BB86FC", "#F4A261"]
RC = "#C8C4FF"   # result vector
SC = "#EF9F27"   # scalar
UC = "#06D6A0"   # unit
PC = "#FFD166"   # projection
DC = "#FF6B9D"   # dot / angle arc
OC = "#B0E0FF"   # orthogonal perpendicular

SCALE    = 44    # px per grid unit
HIT_R    = 13    # drag handle radius (px)
AH_LEN   = 14   # arrowhead shaft length
AH_W     =  9   # arrowhead half-width

OPS = [
    ("none",   "None"),
    ("add",    "Add  v+v"),
    ("sub",    "Sub  v-v"),
    ("scalar", "Scalar kv"),
    ("unit",   "Unit  v̂"),
    ("dot",    "Dot  v·v"),
    ("ortho",  "Orthogonal"),
    ("proj",   "Projection"),
]


# ── Data ──────────────────────────────────────────────────────────────────────
class Vec:
    def __init__(self, x=2.0, y=2.0, color="#4FC3F7"):
        self.x = float(x)
        self.y = float(y)
        self.color = color

    def mag(self):
        return math.hypot(self.x, self.y)

    def dot(self, other):
        return self.x * other.x + self.y * other.y

    def unit(self):
        m = self.mag()
        return (self.x / m, self.y / m) if m > 1e-9 else (0.0, 0.0)


# ── App ───────────────────────────────────────────────────────────────────────
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("2D Vector Playground")
        self.configure(bg=PANEL_BG)
        self.resizable(True, True)

        self.vecs   = [Vec(3, 2, COLORS[0]), Vec(-1, 3, COLORS[1])]
        self.op     = "none"
        self.k      = 2.0
        self.ia     = 0    # index a
        self.ib     = 1    # index b
        self.it     = 0    # scalar / unit target index
        self._drag  = None
        self._rows  = []   # list of dict with tk widgets

        self._build_layout()
        self._rebuild_vec_panel()
        self.after(80, self._draw)

    # ── UI ────────────────────────────────────────────────────────────────────
    def _build_layout(self):
        self.columnconfigure(0, weight=1)
        self.columnconfigure(1, weight=0)
        self.rowconfigure(0, weight=1)

        self.canvas = tk.Canvas(self, bg=BG, highlightthickness=0,
                                 width=580, height=530)
        self.canvas.grid(row=0, column=0, sticky="nsew",
                          padx=(10, 4), pady=10)
        self.canvas.bind("<Configure>",       lambda _: self._draw())
        self.canvas.bind("<ButtonPress-1>",   self._on_press)
        self.canvas.bind("<B1-Motion>",       self._on_drag)
        self.canvas.bind("<ButtonRelease-1>", self._on_release)

        # Scrollable right panel
        self._panel = tk.Frame(self, bg=PANEL_BG, width=235)
        self._panel.grid(row=0, column=1, sticky="ns", padx=(4, 10), pady=10)
        self._panel.grid_propagate(False)
        self._panel.columnconfigure(0, weight=1)

        self._sec("VECTORS")
        self._rows_frame = tk.Frame(self._panel, bg=PANEL_BG)
        self._rows_frame.pack(fill="x", padx=6)

        tk.Button(self._panel, text="+ add vector",
                  bg=BTN_BG, fg=FG_DIM, relief="flat",
                  font=("Courier", 10), cursor="hand2",
                  activebackground="#2A2A55", activeforeground=FG,
                  command=self._add_vec, pady=4
                  ).pack(fill="x", padx=8, pady=(4, 6))

        tk.Frame(self._panel, bg=AXIS_COL, height=1).pack(fill="x",
                                                            padx=8, pady=4)

        self._sec("OPERATION")
        og = tk.Frame(self._panel, bg=PANEL_BG)
        og.pack(fill="x", padx=6, pady=(0, 6))
        og.columnconfigure(0, weight=1)
        og.columnconfigure(1, weight=1)
        self._op_btns = {}
        for idx, (key, label) in enumerate(OPS):
            b = tk.Button(og, text=label, bg=BTN_BG, fg=FG_DIM,
                           relief="flat", font=("Courier", 9),
                           cursor="hand2", pady=4,
                           activebackground=BTN_ON_BG,
                           activeforeground=BTN_ON_FG,
                           command=lambda k=key: self._set_op(k))
            b.grid(row=idx // 2, column=idx % 2, sticky="ew", padx=2, pady=2)
            self._op_btns[key] = b
        self._refresh_op_btns()

        tk.Frame(self._panel, bg=AXIS_COL, height=1).pack(fill="x",
                                                            padx=8, pady=4)

        # Vector selectors (a / b)
        self._ab_frame = tk.Frame(self._panel, bg=PANEL_BG)
        sf = self._ab_frame
        tk.Label(sf, text="a:", bg=PANEL_BG, fg=FG_DIM,
                 font=("Courier", 10)).grid(row=0, column=0, padx=(0, 2))
        self._combo_a = tk.OptionMenu(sf, tk.StringVar(), "")
        self._style_combo(self._combo_a)
        self._combo_a.grid(row=0, column=1, padx=(0, 6), sticky="w")
        tk.Label(sf, text="b:", bg=PANEL_BG, fg=FG_DIM,
                 font=("Courier", 10)).grid(row=0, column=2, padx=(0, 2))
        self._combo_b = tk.OptionMenu(sf, tk.StringVar(), "")
        self._style_combo(self._combo_b)
        self._combo_b.grid(row=0, column=3, sticky="w")
        self._var_a = tk.IntVar(value=0)
        self._var_b = tk.IntVar(value=1)

        # Target selector (scalar / unit)
        self._t_frame = tk.Frame(self._panel, bg=PANEL_BG)
        tk.Label(self._t_frame, text="vector:", bg=PANEL_BG, fg=FG_DIM,
                 font=("Courier", 10)).grid(row=0, column=0, padx=(0, 4))
        self._combo_t = tk.OptionMenu(self._t_frame, tk.StringVar(), "")
        self._style_combo(self._combo_t)
        self._combo_t.grid(row=0, column=1, sticky="w")
        self._var_t = tk.IntVar(value=0)

        # Scalar k
        self._k_frame = tk.Frame(self._panel, bg=PANEL_BG)
        tk.Label(self._k_frame, text="k =", bg=PANEL_BG, fg=FG_DIM,
                 font=("Courier", 10)).grid(row=0, column=0, padx=(0, 4))
        self._k_var = tk.DoubleVar(value=2.0)
        self._k_slider = tk.Scale(
            self._k_frame, from_=-4, to=4, resolution=0.1,
            orient="horizontal", variable=self._k_var,
            bg=PANEL_BG, fg=SC, troughcolor=INPUT_BG,
            highlightthickness=0, relief="flat",
            command=lambda _: self._on_k()
        )
        self._k_slider.grid(row=0, column=1, sticky="ew")
        self._k_frame.columnconfigure(1, weight=1)
        self._k_entry = tk.Entry(
            self._k_frame, textvariable=self._k_var, width=6,
            bg=INPUT_BG, fg=FG, insertbackground=FG,
            relief="flat", font=("Courier", 10),
            highlightthickness=1, highlightbackground=AXIS_COL
        )
        self._k_entry.grid(row=0, column=2, padx=(4, 0))
        self._k_entry.bind("<Return>", lambda _: self._on_k())
        self._k_entry.bind("<FocusOut>", lambda _: self._on_k())

        tk.Frame(self._panel, bg=AXIS_COL, height=1).pack(fill="x",
                                                            padx=8, pady=4)
        self._sec("RESULT")
        self._result_var = tk.StringVar(value="select an operation")
        self._result_lbl = tk.Label(
            self._panel, textvariable=self._result_var,
            bg=INPUT_BG, fg=FG_DIM, relief="flat",
            font=("Courier", 9), justify="left", anchor="nw",
            wraplength=210, padx=6, pady=5
        )
        self._result_lbl.pack(fill="x", padx=8, pady=(0, 8))

        tk.Label(self._panel, text="drag arrowheads on canvas",
                 bg=PANEL_BG, fg=FG_DIM, font=("Courier", 8)
                 ).pack(padx=8, pady=(0, 6))

    def _sec(self, text):
        tk.Label(self._panel, text=text, bg=PANEL_BG, fg=FG_DIM,
                 font=("Courier", 9), anchor="w"
                 ).pack(fill="x", padx=10, pady=(8, 3))

    def _style_combo(self, w):
        w.configure(bg=INPUT_BG, fg=FG, activebackground=BTN_ON_BG,
                    activeforeground=BTN_ON_FG, relief="flat",
                    font=("Courier", 9), highlightthickness=0, width=4)
        w["menu"].configure(bg=INPUT_BG, fg=FG, activebackground=BTN_ON_BG,
                             activeforeground=BTN_ON_FG)

    # ── Op panel helpers ──────────────────────────────────────────────────────
    def _set_op(self, key):
        self.op = key
        self._refresh_op_btns()
        self._update_sub_panels()
        self._draw()

    def _refresh_op_btns(self):
        for key, btn in self._op_btns.items():
            if key == self.op:
                btn.configure(bg=BTN_ON_BG, fg=BTN_ON_FG)
            else:
                btn.configure(bg=BTN_BG, fg=FG_DIM)

    def _update_sub_panels(self):
        ab_ops = {"add", "sub", "dot", "ortho", "proj"}
        t_ops  = {"scalar", "unit"}
        if self.op in ab_ops:
            self._ab_frame.pack(fill="x", padx=8, pady=(0, 5))
        else:
            self._ab_frame.pack_forget()
        if self.op in t_ops:
            self._t_frame.pack(fill="x", padx=8, pady=(0, 5))
        else:
            self._t_frame.pack_forget()
        if self.op == "scalar":
            self._k_frame.pack(fill="x", padx=8, pady=(0, 5))
        else:
            self._k_frame.pack_forget()

    def _on_k(self):
        try:
            self.k = float(self._k_var.get())
        except Exception:
            pass
        self._draw()

    # ── Vector panel (rows) ───────────────────────────────────────────────────
    def _rebuild_vec_panel(self):
        for r in self._rows:
            r["frame"].destroy()
        self._rows.clear()

        for i, v in enumerate(self.vecs):
            fr = tk.Frame(self._rows_frame, bg=PANEL_BG)
            fr.pack(fill="x", pady=3)
            fr.columnconfigure(2, weight=1)
            fr.columnconfigure(3, weight=1)

            dot = tk.Canvas(fr, width=10, height=10,
                             bg=PANEL_BG, highlightthickness=0)
            dot.create_oval(1, 1, 9, 9, fill=v.color, outline="")
            dot.grid(row=0, column=0, padx=(0, 4))

            tk.Label(fr, text=f"v{i+1}", bg=PANEL_BG, fg=v.color,
                     font=("Courier", 10, "bold")).grid(row=0, column=1,
                                                         padx=(0, 4))

            xv = tk.StringVar(value=f"{v.x:.1f}")
            yv = tk.StringVar(value=f"{v.y:.1f}")

            def mk_e(parent, var, col, vc=v.color):
                e = tk.Entry(parent, textvariable=var, width=6,
                              bg=INPUT_BG, fg=FG, insertbackground=FG,
                              relief="flat", font=("Courier", 10),
                              highlightthickness=1,
                              highlightbackground=AXIS_COL,
                              highlightcolor=vc)
                e.grid(row=0, column=col, padx=2)
                return e

            mk_e(fr, xv, 2)
            mk_e(fr, yv, 3)

            tk.Button(fr, text="×", bg=PANEL_BG, fg=FG_DIM,
                      relief="flat", font=("Courier", 12), cursor="hand2",
                      activebackground=PANEL_BG, activeforeground="#EF476F",
                      command=lambda idx=i: self._del_vec(idx)
                      ).grid(row=0, column=4, padx=(2, 0))

            def mk_cb(idx, xvar, yvar):
                def cb(*_):
                    try:
                        self.vecs[idx].x = float(xvar.get())
                        self.vecs[idx].y = float(yvar.get())
                        self._draw()
                    except ValueError:
                        pass
                return cb

            cb = mk_cb(i, xv, yv)
            xv.trace_add("write", cb)
            yv.trace_add("write", cb)
            self._rows.append({"frame": fr, "xv": xv, "yv": yv})

        self._rebuild_combos()

    def _rebuild_combos(self):
        n = len(self.vecs)
        opts = [f"v{i+1}" for i in range(n)]

        def rebuild_menu(combo, var, cur_idx, default=0):
            menu = combo["menu"]
            menu.delete(0, "end")
            for j, label in enumerate(opts):
                menu.add_command(
                    label=label,
                    command=lambda j=j, v=var: (v.set(j), self._on_sel())
                )
            safe = min(cur_idx, n - 1)
            var.set(safe)

        rebuild_menu(self._combo_a, self._var_a, self._var_a.get())
        rebuild_menu(self._combo_b, self._var_b,
                     max(1, self._var_b.get()), default=1)
        rebuild_menu(self._combo_t, self._var_t, self._var_t.get())

        # Relabel the OptionMenus to show current selection
        for combo, var in [(self._combo_a, self._var_a),
                            (self._combo_b, self._var_b),
                            (self._combo_t, self._var_t)]:
            idx = min(var.get(), n - 1)
            combo.configure(text=f"v{idx+1}")

    def _on_sel(self):
        # Update labels
        for combo, var in [(self._combo_a, self._var_a),
                            (self._combo_b, self._var_b),
                            (self._combo_t, self._var_t)]:
            n = len(self.vecs)
            idx = min(var.get(), n - 1)
            combo.configure(text=f"v{idx+1}")
        self._draw()

    def _sync_entries(self):
        for i, (v, r) in enumerate(zip(self.vecs, self._rows)):
            xs = f"{v.x:.1f}"; ys = f"{v.y:.1f}"
            if r["xv"].get() != xs: r["xv"].set(xs)
            if r["yv"].get() != ys: r["yv"].set(ys)

    def _add_vec(self):
        if len(self.vecs) >= 6:
            return
        self.vecs.append(Vec(1, 2, COLORS[len(self.vecs) % len(COLORS)]))
        self._rebuild_vec_panel()
        self._draw()

    def _del_vec(self, idx):
        self.vecs.pop(idx)
        self._rebuild_vec_panel()
        self._draw()

    # ── Coordinate conversion ─────────────────────────────────────────────────
    def _origin(self):
        w = self.canvas.winfo_width()
        h = self.canvas.winfo_height()
        return w / 2, h / 2

    def _to_px(self, vx, vy):
        ox, oy = self._origin()
        return ox + vx * SCALE, oy - vy * SCALE

    def _to_vec(self, px, py):
        ox, oy = self._origin()
        return (px - ox) / SCALE, -(py - oy) / SCALE

    def _clamp(self, i):
        return max(0, min(i, len(self.vecs) - 1))

    # ── Canvas drawing primitives ─────────────────────────────────────────────
    def _arrow(self, x0, y0, x1, y1, color, width=2, dash=()):
        dx = x1 - x0; dy = y1 - y0
        L = math.hypot(dx, dy)
        if L < 2:
            return
        ux = dx / L; uy = dy / L
        sx = x1 - ux * AH_LEN; sy = y1 - uy * AH_LEN
        kw = dict(fill=color, width=width, capstyle="round")
        if dash:
            kw["dash"] = dash
        self.canvas.create_line(x0, y0, sx, sy, **kw)
        self.canvas.create_polygon(
            x1, y1,
            sx - uy * AH_W, sy + ux * AH_W,
            sx + uy * AH_W, sy - ux * AH_W,
            fill=color, outline=""
        )

    def _dashed(self, x0, y0, x1, y1, color, width=1):
        self.canvas.create_line(x0, y0, x1, y1,
                                 fill=color, dash=(4, 4), width=width)

    def _handle(self, tx, ty, color):
        r = 5
        self.canvas.create_oval(tx - r, ty - r, tx + r, ty + r,
                                  outline=color, fill=BG, width=2)

    def _text(self, x, y, text, color, size=9):
        self.canvas.create_text(x, y, text=text, fill=color,
                                  font=("Courier", size, "bold"), anchor="w")

    def _set_result(self, lines):
        self._result_var.set("\n".join(lines))

    # ── Main draw ─────────────────────────────────────────────────────────────
    def _draw(self):
        cv = self.canvas
        cv.delete("all")
        W = cv.winfo_width(); H = cv.winfo_height()
        if W < 4 or H < 4:
            return

        ox, oy = self._origin()
        rng = int(max(W, H) / SCALE) + 2

        # Grid
        for i in range(-rng, rng + 1):
            gx = ox + i * SCALE; gy = oy - i * SCALE
            col = GRID_MAJ if i % 5 == 0 else GRID_MIN
            cv.create_line(gx, 0, gx, H, fill=col, width=1)
            cv.create_line(0, gy, W, gy, fill=col, width=1)

        # Axes
        cv.create_line(0, oy, W, oy, fill=AXIS_COL, width=1.5)
        cv.create_line(ox, 0, ox, H, fill=AXIS_COL, width=1.5)

        # Tick labels
        for i in range(-rng, rng + 1):
            if i == 0:
                continue
            gx = ox + i * SCALE; gy = oy - i * SCALE
            if 4 < gx < W - 4:
                cv.create_text(gx, oy + 14, text=str(i),
                                fill=FG_DIM, font=("Courier", 8))
            if 4 < gy < H - 4:
                cv.create_text(ox - 16, gy, text=str(i),
                                fill=FG_DIM, font=("Courier", 8))

        self._draw_op(ox, oy, W, H)
        self._draw_base(ox, oy)

    # ── Operation rendering ───────────────────────────────────────────────────
    def _draw_op(self, ox, oy, W, H):
        op = self.op
        V  = self.vecs
        if not V:
            return
        ia = self._clamp(self._var_a.get())
        ib = self._clamp(self._var_b.get())
        it = self._clamp(self._var_t.get())
        k  = self.k

        # ── None ──
        if op == "none":
            self._set_result(["No operation selected.",
                               "Click a button above."])
            return

        # ── Add / Sub ──
        if op in ("add", "sub"):
            a, b = V[ia], V[ib]
            rx = a.x + b.x if op == "add" else a.x - b.x
            ry = a.y + b.y if op == "add" else a.y - b.y
            rtx, rty = self._to_px(rx, ry)
            ax, ay   = self._to_px(a.x, a.y)
            bx, by   = self._to_px(b.x, b.y)
            mg = math.hypot(rx, ry)
            if op == "add":
                self.canvas.create_polygon(
                    ox, oy, ax, ay, rtx, rty, bx, by,
                    fill="#ffffff", outline="", stipple="gray12"
                )
                self._dashed(ax, ay, rtx, rty, b.color)
                self._dashed(bx, by, rtx, rty, a.color)
                self._arrow(ax, ay, rtx, rty, b.color, width=1)
            else:
                self._arrow(ax, ay, rtx, rty, b.color, width=1)
            self._arrow(ox, oy, rtx, rty, RC, width=3)
            self._text(rtx + 8, rty - 8,
                        f"=[{rx:.1f},{ry:.1f}] |{mg:.2f}|", RC)
            self._handle(rtx, rty, RC)
            self._set_result([
                "Addition" if op == "add" else "Subtraction",
                f"a  = [{a.x:.1f}, {a.y:.1f}]",
                f"b  = [{b.x:.1f}, {b.y:.1f}]",
                f"   = [{rx:.2f}, {ry:.2f}]",
                f"|r|= {mg:.4f}",
            ])

        # ── Scalar ──
        elif op == "scalar":
            v = V[it]
            rx, ry = k * v.x, k * v.y
            ptx, pty = self._to_px(rx, ry)
            vtx, vty = self._to_px(v.x, v.y)
            self._arrow(ox, oy, vtx, vty, v.color, width=1)  # faint original
            self._arrow(ox, oy, ptx, pty, SC, width=3)
            mg = math.hypot(rx, ry)
            self._text(ptx + 8, pty - 8,
                        f"{k:.1f}v=[{rx:.1f},{ry:.1f}] |{mg:.2f}|", SC)
            self._handle(ptx, pty, SC)
            self._draw_k_line(k, W, H, ox, oy)
            self._set_result([
                "Scalar Multiplication",
                f"k   = {k:.3f}",
                f"v   = [{v.x:.1f}, {v.y:.1f}]",
                f"kv  = [{rx:.2f}, {ry:.2f}]",
                f"|kv|= {mg:.4f}",
                f"|v| = {v.mag():.4f}",
            ])

        # ── Unit vector ──
        elif op == "unit":
            v = V[it]; m = v.mag()
            if m > 1e-9:
                ux, uy = v.unit()
                utx, uty = self._to_px(ux, uy)
                vtx, vty = self._to_px(v.x, v.y)
                r_px = SCALE
                self.canvas.create_oval(
                    ox - r_px, oy - r_px, ox + r_px, oy + r_px,
                    outline=UC, width=1, dash=(3, 4)
                )
                self._arrow(ox, oy, vtx, vty, v.color, width=1)  # faint
                self._arrow(ox, oy, utx, uty, UC, width=3)
                self.canvas.create_oval(utx - 5, uty - 5,
                                         utx + 5, uty + 5, fill=UC, outline="")
                self._text(utx + 8, uty - 8,
                            f"v̂=[{ux:.3f},{uy:.3f}]", UC)
                ang = math.degrees(math.atan2(uy, ux))
                self._set_result([
                    "Unit Vector",
                    f"v  = [{v.x:.1f}, {v.y:.1f}]",
                    f"|v|= {m:.4f}",
                    f"v̂  = [{ux:.4f},",
                    f"       {uy:.4f}]",
                    f"θ  = {ang:.2f}°",
                    "|v̂| = 1.0000",
                ])
            else:
                self._set_result(["Unit Vector",
                                   "Zero vector — undefined."])

        # ── Dot product ──
        elif op == "dot":
            a, b = V[ia], V[ib]
            d = a.dot(b)
            ma, mb2 = a.mag(), b.mag()
            ang = 0.0
            if ma > 1e-9 and mb2 > 1e-9:
                cos_t = max(-1.0, min(1.0, d / (ma * mb2)))
                ang = math.degrees(math.acos(cos_t))
            a1 = math.atan2(-a.y, a.x)
            a2 = math.atan2(-b.y, b.x)
            da = a2 - a1
            while da >  math.pi: da -= 2 * math.pi
            while da < -math.pi: da += 2 * math.pi
            r_arc = int(SCALE * 0.55)
            pts = []
            for s in range(37):
                ang2 = a1 + da * s / 36
                pts += [ox + r_arc * math.cos(ang2),
                         oy + r_arc * math.sin(ang2)]
            if len(pts) >= 4:
                self.canvas.create_line(*pts, fill=DC, width=1.5, smooth=True)
            amid = a1 + da / 2
            self.canvas.create_text(
                ox + (r_arc + 14) * math.cos(amid),
                oy + (r_arc + 14) * math.sin(amid),
                text=f"{ang:.1f}°", fill=DC, font=("Courier", 8, "bold")
            )
            self._set_result([
                "Dot Product",
                f"a  = [{a.x:.1f}, {a.y:.1f}]",
                f"b  = [{b.x:.1f}, {b.y:.1f}]",
                f"a·b= {d:.4f}",
                f"θ  = {ang:.3f}°",
                f"|a|= {ma:.4f}",
                f"|b|= {mb2:.4f}",
            ])

        # ── Orthogonal ──
        elif op == "ortho":
            a, b = V[ia], V[ib]
            d = a.dot(b)
            ma, mb2 = a.mag(), b.mag()
            oa_x, oa_y = -a.y, a.x
            oatx, oaty = self._to_px(oa_x, oa_y)
            self._arrow(ox, oy, oatx, oaty, OC, width=2, dash=(5, 4))
            self._text(oatx + 8, oaty - 8,
                        f"a⊥=[{oa_x:.1f},{oa_y:.1f}]", OC)
            if ma > 1e-9:
                sz = 12
                ux2 = a.x / ma; uy2 = a.y / ma
                vx2 = -uy2;     vy2 = ux2
                self.canvas.create_line(
                    ox + ux2 * sz,            oy - uy2 * sz,
                    ox + ux2 * sz + vx2 * sz, oy - uy2 * sz - vy2 * sz,
                    ox + vx2 * sz,            oy - vy2 * sz,
                    fill="white", width=1, dash=(2, 2)
                )
            is_ortho = abs(d) < 0.05
            if is_ortho:
                self.canvas.create_text(
                    ox - SCALE * 2.5, oy - SCALE * 2.8,
                    text="✓ orthogonal!", fill=UC,
                    font=("Courier", 10, "bold")
                )
            ang = 0.0
            if ma > 1e-9 and mb2 > 1e-9:
                cos_t = max(-1.0, min(1.0, d / (ma * mb2)))
                ang = math.degrees(math.acos(cos_t))
            self._set_result([
                "Orthogonal",
                f"a   = [{a.x:.1f}, {a.y:.1f}]",
                f"b   = [{b.x:.1f}, {b.y:.1f}]",
                f"a⊥  = [{oa_x:.1f}, {oa_y:.1f}]",
                f"a·b = {d:.4f}",
                f"θ   = {ang:.2f}°",
                "✓ orthogonal" if is_ortho else "not orthogonal",
            ])

        # ── Projection ──
        elif op == "proj":
            a, b = V[ia], V[ib]
            mb2 = b.mag()
            if mb2 > 1e-9:
                s = a.dot(b) / (mb2 * mb2)
                px2, py2 = s * b.x, s * b.y
                projx, projy = self._to_px(px2, py2)
                ax, ay = self._to_px(a.x, a.y)
                # Extended b-axis
                bex, bey = self._to_px(b.x * 2.5, b.y * 2.5)
                bnx, bny = self._to_px(-b.x * 0.4, -b.y * 0.4)
                self._dashed(bnx, bny, bex, bey, b.color)
                # Perpendicular drop
                self.canvas.create_line(ax, ay, projx, projy,
                                         fill="white", width=1, dash=(4, 3))
                # Right-angle box
                bux = b.x / mb2; buy = b.y / mb2
                perp_x = ax - projx; perp_y = ay - projy
                pl = math.hypot(perp_x, perp_y)
                if pl > 1e-9:
                    pux = perp_x / pl; puy = perp_y / pl
                    sz = 10
                    self.canvas.create_line(
                        projx + bux * sz,            projy - buy * sz,
                        projx + bux * sz + pux * sz, projy - buy * sz + puy * sz,
                        projx + pux * sz,            projy + puy * sz,
                        fill="white", width=1
                    )
                self._arrow(ox, oy, projx, projy, PC, width=3)
                pmg = math.hypot(px2, py2)
                self._text(projx + 8, projy - 8,
                            f"proj=[{px2:.2f},{py2:.2f}]", PC)
                self._handle(projx, projy, PC)
                self._set_result([
                    "Projection of a onto b",
                    f"a      = [{a.x:.1f}, {a.y:.1f}]",
                    f"b      = [{b.x:.1f}, {b.y:.1f}]",
                    f"scalar = {s:.4f}",
                    f"proj   = [{px2:.3f},",
                    f"           {py2:.3f}]",
                    f"|proj| = {pmg:.4f}",
                ])
            else:
                self._set_result(["Projection", "b is the zero vector."])

    # ── Scalar number line ────────────────────────────────────────────────────
    def _draw_k_line(self, kv, W, H, ox, oy):
        lx = 24; rx = W - 24; y = H - 28
        cx = (lx + rx) / 2
        u  = (rx - lx) / 10

        self.canvas.create_line(lx, y, rx, y, fill=AXIS_COL, width=2)
        for i in range(-5, 6):
            tx = cx + i * u
            self.canvas.create_line(tx, y - 4, tx, y + 4,
                                     fill=AXIS_COL, width=1)
            if i % 2 == 0:
                self.canvas.create_text(tx, y + 13, text=str(i),
                                         fill=FG_DIM, font=("Courier", 8))
        cl = max(-4.0, min(4.0, kv))
        if abs(cl) > 0.01:
            self.canvas.create_rectangle(
                cx, y - 5, cx + cl * u, y + 5,
                fill=SC, outline="", stipple="gray25"
            )
        dx = cx + cl * u
        self.canvas.create_oval(dx - 7, y - 7, dx + 7, y + 7,
                                  fill=SC, outline="")
        self.canvas.create_text(dx, y, text=f"{kv:.1f}",
                                  fill="#111111",
                                  font=("Courier", 7, "bold"))
        self.canvas.create_text(lx, y - 14, text="scalar  k",
                                  fill=FG_DIM, font=("Courier", 8),
                                  anchor="w")

    # ── Base vectors ──────────────────────────────────────────────────────────
    def _draw_base(self, ox, oy):
        it = self._clamp(self._var_t.get())
        for i, v in enumerate(self.vecs):
            tx, ty = self._to_px(v.x, v.y)
            # Component dashes
            self.canvas.create_line(ox, oy, tx, oy,
                                     fill=v.color, dash=(4, 4), width=1)
            self.canvas.create_line(tx, oy, tx, ty,
                                     fill=v.color, dash=(4, 4), width=1)
            # Dim target vector when scalar/unit op is active
            dim = (self.op in ("scalar", "unit") and i != it)
            self._arrow(ox, oy, tx, ty, v.color, width=1 if dim else 2)
            self.canvas.create_text(
                tx + 8, ty - 8,
                text=f"v{i+1} [{v.x:.1f},{v.y:.1f}]",
                fill=v.color, font=("Courier", 9, "bold"), anchor="w"
            )
            self._handle(tx, ty, v.color)

    # ── Mouse ─────────────────────────────────────────────────────────────────
    def _on_press(self, event):
        for i in range(len(self.vecs) - 1, -1, -1):
            tx, ty = self._to_px(self.vecs[i].x, self.vecs[i].y)
            if math.hypot(event.x - tx, event.y - ty) < HIT_R:
                self._drag = i
                return

    def _on_drag(self, event):
        if self._drag is None:
            return
        vx, vy = self._to_vec(event.x, event.y)
        self.vecs[self._drag].x = round(vx * 10) / 10
        self.vecs[self._drag].y = round(vy * 10) / 10
        self._sync_entries()
        self._draw()

    def _on_release(self, _event):
        self._drag = None


if __name__ == "__main__":
    App().mainloop()